In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent

sys.path.insert(0, str(ROOT))

from scripts.evaluation.run_correlations import run_correlations
from scripts.evaluation.run_regression import vif_table, run_regression
from scripts.evaluation.run_loocv import loocv_summary

### *Constants*

In [2]:
TRANSLUCENT = [
    "complexity_avg_raw", 
    "complexity_avg_norm", 
    "complexity_div", 
    "complexity_disp",  
    "complexity_diss_aa_seq",
]

BASELINE = [
    # Complexity.perform_measurements
    "magnitude",
    "support",
    "variety",
    "level_of_detail",
    "time_granularity",
    "structure",
    "affinity",
    "trace_length_min",
    "trace_length_avg",
    "trace_length_max",
    "distinct_traces",
    "deviation_from_random",
    "lempel_ziv_complexity",
    "pentland_task_complexity",
    "pentland_process_complexity",
    # Complexity.graph_complexity
    "variant_entropy",
    "variant_entropy_norm",
    # Complexity.log_complexity
    "sequence_entropy",
    "sequence_entropy_norm",
    "sequence_entropy_linear",
    "sequence_entropy_linear_norm",
    "sequence_entropy_exp",
    "sequence_entropy_exp_norm",
]

QUALITY = [
    "f_score_imtf", 
    "f_score_imts", 
    "f_score_imto", 
    "f_score_imftf", 
    "f_score_imfts", 
    "f_score_imfto",
]

CONTROL = [
    "num_activities"
]

### *Group 1 (real_life and synthetic)*

In [3]:
from functools import reduce

import pandas as pd

FILES = ["complexity", "quality", "baseline_complexity", "activity_number"]


def load_dataset(folder: str) -> pd.DataFrame:
    """Read the four result files of one dataset and inner-join them on the log name."""
    path = ROOT / "results" / "evaluation" / folder
    dfs = [pd.read_csv(path / f"{f}.csv") for f in FILES]
    df = reduce(lambda left, right: left.merge(right, on="name", how="inner", validate="1:1"), dfs)
    assert len(df) == len(dfs[0]), f"{folder}: inner join dropped logs"
    df = df.drop(columns=df.filter(regex=r"_(std|z)$").columns)
    return df.assign(dataset=folder)


# real-life and synthetic data (raw values, never modified)
rl_df = load_dataset("real_life")
sy_df = load_dataset("synthetic")
group1_raw_df = pd.concat([rl_df, sy_df], ignore_index=True)

print(group1_raw_df.shape)
print("\n")
print(group1_raw_df.columns)

(39, 54)


Index(['name', 'complexity_avg_raw', 'complexity_avg_norm', 'complexity_div',
       'complexity_disp', 'complexity_diss_aa_seq', 'complexity_diss_aa_mset',
       'complexity_diss_ac_seq', 'complexity_diss_ac_mset', 'precision_imts',
       'fitness_imts', 'f_score_imts', 'precision_imto', 'fitness_imto',
       'f_score_imto', 'precision_imtf', 'fitness_imtf', 'f_score_imtf',
       'precision_imfts', 'fitness_imfts', 'f_score_imfts', 'precision_imfto',
       'fitness_imfto', 'f_score_imfto', 'precision_imftf', 'fitness_imftf',
       'f_score_imftf', 'magnitude', 'support', 'variety', 'level_of_detail',
       'time_granularity', 'structure', 'affinity', 'trace_length_min',
       'trace_length_avg', 'trace_length_max', 'distinct_traces',
       'deviation_from_random', 'lempel_ziv_complexity',
       'pentland_task_complexity', 'pentland_process_complexity',
       'variant_entropy', 'variant_entropy_norm', 'sequence_entropy',
       'sequence_entropy_norm', 'sequence_e

### *Group 2 (simulated)*

In [4]:
# simulated data (raw values, never modified)
group2_raw_df = load_dataset("simulated")

print(group2_raw_df.shape)
print("\n")
print(group2_raw_df.columns)

(30, 54)


Index(['name', 'complexity_avg_raw', 'complexity_avg_norm', 'complexity_div',
       'complexity_disp', 'complexity_diss_aa_seq', 'complexity_diss_aa_mset',
       'complexity_diss_ac_seq', 'complexity_diss_ac_mset', 'precision_imts',
       'fitness_imts', 'f_score_imts', 'precision_imto', 'fitness_imto',
       'f_score_imto', 'precision_imtf', 'fitness_imtf', 'f_score_imtf',
       'precision_imfts', 'fitness_imfts', 'f_score_imfts', 'precision_imfto',
       'fitness_imfto', 'f_score_imfto', 'precision_imftf', 'fitness_imftf',
       'f_score_imftf', 'magnitude', 'support', 'variety', 'level_of_detail',
       'time_granularity', 'structure', 'affinity', 'trace_length_min',
       'trace_length_avg', 'trace_length_max', 'distinct_traces',
       'deviation_from_random', 'lempel_ziv_complexity',
       'pentland_task_complexity', 'pentland_process_complexity',
       'variant_entropy', 'variant_entropy_norm', 'sequence_entropy',
       'sequence_entropy_norm', 'sequence_e

### *Combined (real-life, synthetic, and simulated)*

In [5]:
def standardize(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of df with the translucent measures z-scored over the logs in df.

    Computed from the raw values, over the sample the analysis is run on (group 1, group 2
    or all logs), so that datasets are not standardized separately. ddof=0 as StandardScaler.
    Baseline measures stay raw: correlations do not depend on a linear rescaling."""
    df = df.copy()
    x = df[TRANSLUCENT]
    df[TRANSLUCENT] = (x - x.mean()) / x.std(ddof=0)
    return df


# built from the raw frames, so running this cell again gives the same result
group1_df = standardize(group1_raw_df)
group2_df = standardize(group2_raw_df)
all_df = standardize(pd.concat([group1_raw_df, group2_raw_df], ignore_index=True))

print(all_df.shape)
print(all_df.columns)

(69, 54)
Index(['name', 'complexity_avg_raw', 'complexity_avg_norm', 'complexity_div',
       'complexity_disp', 'complexity_diss_aa_seq', 'complexity_diss_aa_mset',
       'complexity_diss_ac_seq', 'complexity_diss_ac_mset', 'precision_imts',
       'fitness_imts', 'f_score_imts', 'precision_imto', 'fitness_imto',
       'f_score_imto', 'precision_imtf', 'fitness_imtf', 'f_score_imtf',
       'precision_imfts', 'fitness_imfts', 'f_score_imfts', 'precision_imfto',
       'fitness_imfto', 'f_score_imfto', 'precision_imftf', 'fitness_imftf',
       'f_score_imftf', 'magnitude', 'support', 'variety', 'level_of_detail',
       'time_granularity', 'structure', 'affinity', 'trace_length_min',
       'trace_length_avg', 'trace_length_max', 'distinct_traces',
       'deviation_from_random', 'lempel_ziv_complexity',
       'pentland_task_complexity', 'pentland_process_complexity',
       'variant_entropy', 'variant_entropy_norm', 'sequence_entropy',
       'sequence_entropy_norm', 'sequence_ent

### *Analysis*

In [6]:
T_COLUMNS = ["name", *TRANSLUCENT, *QUALITY, *CONTROL]
B_COLUMNS = ["name", *BASELINE, *QUALITY, *CONTROL]

#### Correlations - group 1

In [7]:
OUTPUT = ROOT / "results" / "evaluation" / "group1_corr"
outputs = [OUTPUT / "p_corr.csv", OUTPUT / "p_p_values.csv", OUTPUT / "k_corr.csv", OUTPUT / "k_p_values.csv"]

_, _, _, _ = run_correlations(group1_df[T_COLUMNS], outputs, "translucent")
_, _, _, _ = run_correlations(group1_df[B_COLUMNS], outputs, "baseline")

wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\group1_corr\p_corr_translucent.csv
wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\group1_corr\p_p_values_translucent.csv
wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\group1_corr\k_corr_translucent.csv
wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\group1_corr\k_p_values_translucent.csv
wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\group1_corr\p_corr_baseline.csv
wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\group1_corr\p_p_values_baseline.csv
wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\group1_corr\k_corr_baseline.csv
wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\group1_corr\k_p_values_baseline.csv


#### Correlations - group 2

In [8]:
OUTPUT = ROOT / "results" / "evaluation" / "group2_corr"
outputs = [OUTPUT / "p_corr.csv", OUTPUT / "p_p_values.csv", OUTPUT / "k_corr.csv", OUTPUT / "k_p_values.csv"]

_, _, _, _ = run_correlations(group2_df[T_COLUMNS], outputs, "translucent")
_, _, _, _ = run_correlations(group2_df[B_COLUMNS], outputs, "baseline")

wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\group2_corr\p_corr_translucent.csv
wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\group2_corr\p_p_values_translucent.csv
wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\group2_corr\k_corr_translucent.csv
wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\group2_corr\k_p_values_translucent.csv
wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\group2_corr\p_corr_baseline.csv
wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\group2_corr\p_p_values_baseline.csv
wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\group2_corr\k_corr_baseline.csv
wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\group2_corr\k_p_values_baseline.csv


#### Correlations - combined

In [9]:
OUTPUT = ROOT / "results" / "evaluation" / "corr"
outputs = [OUTPUT / "p_corr.csv", OUTPUT / "p_p_values.csv", OUTPUT / "k_corr.csv", OUTPUT / "k_p_values.csv"]

_, _, _, _ = run_correlations(all_df[T_COLUMNS], outputs, "translucent")
_, _, _, _ = run_correlations(all_df[B_COLUMNS], outputs, "baseline")

wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\corr\p_corr_translucent.csv
wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\corr\p_p_values_translucent.csv
wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\corr\k_corr_translucent.csv
wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\corr\k_p_values_translucent.csv
wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\corr\p_corr_baseline.csv
wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\corr\p_p_values_baseline.csv
wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\corr\k_corr_baseline.csv
wrote C:\Users\jiayuan.chen\Documents\translucent-log-complexity\results\evaluation\corr\k_p_values_baseline.csv


#### Regressions - combined

In [10]:
print(vif_table(all_df, TRANSLUCENT))

                 variable       VIF
1      complexity_avg_raw  1.392315
2     complexity_avg_norm  1.273572
3          complexity_div  1.180380
4         complexity_disp  1.875778
5  complexity_diss_aa_seq  1.563334


In [11]:
print(vif_table(all_df, BASELINE))

                        variable           VIF
1                      magnitude    313.076062
2                        support    170.205354
3                        variety      6.759755
4                level_of_detail      8.925144
5               time_granularity      2.224519
6                      structure      5.630703
7                       affinity      8.334726
8               trace_length_min     11.230050
9               trace_length_avg     22.346504
10              trace_length_max      9.663789
11               distinct_traces     16.971395
12         deviation_from_random      6.077811
13         lempel_ziv_complexity     30.132456
14      pentland_task_complexity     78.286688
15   pentland_process_complexity      3.957219
16               variant_entropy     75.490209
17          variant_entropy_norm      7.420367
18              sequence_entropy  12446.653204
19         sequence_entropy_norm    264.978741
20       sequence_entropy_linear   3270.708969
21  sequence_

In [12]:
REDUCED_BASELINE = [
    "variety",                # size: number of distinct activities
    "trace_length_avg",       # trace length
    "variant_entropy_norm",   # variation: entropy of the variants
    "affinity",               # similarity: overlap between traces
    "sequence_entropy_norm",  # sequence entropy
]

print(vif_table(all_df, REDUCED_BASELINE))

                variable       VIF
1                variety  1.385815
2       trace_length_avg  1.820265
3   variant_entropy_norm  2.417945
4               affinity  2.367107
5  sequence_entropy_norm  3.553043


In [13]:
OUT_DIR = ROOT / "results" / "evaluation" / "reg"
run_regression(all_df, QUALITY, TRANSLUCENT, OUT_DIR, "translucent")


--- f_score_imtf ---
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                      0.8015      0.018     45.306      0.000       0.766       0.837
complexity_avg_raw        -0.0712      0.021     -3.437      0.001      -0.113      -0.030
complexity_avg_norm       -0.0090      0.020     -0.453      0.652      -0.049       0.031
complexity_div            -0.0059      0.019     -0.308      0.759      -0.044       0.032
complexity_disp           -0.0378      0.024     -1.557      0.125      -0.086       0.011
complexity_diss_aa_seq     0.0257      0.022      1.164      0.249      -0.018       0.070
n=68  R2=0.231  adj.R2=0.169  F p=0.0052

--- f_score_imts ---
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const

In [14]:
run_regression(all_df, QUALITY, REDUCED_BASELINE, OUT_DIR, "baseline")


--- f_score_imtf ---
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                     1.1365      0.078     14.616      0.000       0.981       1.292
variety                  -0.0022      0.003     -0.876      0.384      -0.007       0.003
trace_length_avg         -0.0130      0.004     -3.386      0.001      -0.021      -0.005
variant_entropy_norm     -0.0471      0.126     -0.375      0.709      -0.299       0.204
affinity                 -0.1554      0.093     -1.670      0.100      -0.341       0.031
sequence_entropy_norm    -0.4263      0.182     -2.347      0.022      -0.789      -0.063
n=68  R2=0.462  adj.R2=0.419  F p=0.0000

--- f_score_imts ---
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const          

#### *LOOCV - combined*

In [15]:
OUT_DIR = ROOT / "results" / "evaluation" / "reg"

print("--- LOOCV Summary - translucent complexity ---")
loocv_summary(all_df, QUALITY, TRANSLUCENT, OUT_DIR, "translucent")
print("\n")
print("--- LOOCV Summary - baseline complexity ---")
loocv_summary(all_df, QUALITY, REDUCED_BASELINE, OUT_DIR, "baseline")

--- LOOCV Summary - translucent complexity ---
              controls   n  R2_in  adjR2_in     Q2   RMSE
target                                                   
f_score_imtf            68  0.231     0.169  0.120  0.149
f_score_imts            68  0.317     0.262  0.182  0.120
f_score_imto            68  0.210     0.146  0.053  0.207
f_score_imftf           68  0.163     0.096 -0.401  0.185
f_score_imfts           68  0.381     0.331  0.245  0.108
f_score_imfto           68  0.173     0.106 -0.375  0.237


--- LOOCV Summary - baseline complexity ---
              controls   n  R2_in  adjR2_in     Q2   RMSE
target                                                   
f_score_imtf            68  0.462     0.419  0.349  0.128
f_score_imts            68  0.303     0.247  0.091  0.126
f_score_imto            68  0.379     0.329  0.256  0.184
f_score_imftf           68  0.505     0.465  0.391  0.122
f_score_imfts           68  0.365     0.313  0.154  0.114
f_score_imfto           68  0.484    

#### *Controlled Group*

In [17]:
run_regression(all_df, QUALITY, TRANSLUCENT, OUT_DIR, "translucent", controls=CONTROL)
run_regression(all_df, QUALITY, REDUCED_BASELINE, OUT_DIR, "baseline", controls=CONTROL)


--- f_score_imtf ---  (controls: num_activities)
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                      0.8243      0.066     12.510      0.000       0.693       0.956
complexity_avg_raw        -0.0630      0.031     -2.032      0.046      -0.125      -0.001
complexity_avg_norm       -0.0157      0.027     -0.574      0.568      -0.070       0.039
complexity_div            -0.0056      0.019     -0.292      0.771      -0.044       0.033
complexity_disp           -0.0418      0.027     -1.557      0.125      -0.095       0.012
complexity_diss_aa_seq     0.0277      0.023      1.208      0.232      -0.018       0.074
num_activities            -0.0015      0.004     -0.359      0.721      -0.010       0.007
n=68  R2=0.233  adj.R2=0.157  F p=0.0106

--- f_score_imts ---  (controls: num_activities)
                             coef    std

In [19]:
print("--- LOOCV Summary - translucent complexity - controlled ---")
loocv_summary(all_df, QUALITY, TRANSLUCENT, OUT_DIR, "translucent", controls=CONTROL)
print("\n")
print("--- LOOCV Summary - baseline complexity -controlled ---")
loocv_summary(all_df, QUALITY, REDUCED_BASELINE, OUT_DIR, "baseline", controls=CONTROL)

--- LOOCV Summary - translucent complexity - controlled ---
                     controls   n  R2_in  adjR2_in     Q2   RMSE
target                                                          
f_score_imtf   num_activities  68  0.233     0.157  0.101  0.151
f_score_imts   num_activities  68  0.317     0.250  0.161  0.121
f_score_imto   num_activities  68  0.215     0.137  0.033  0.210
f_score_imftf  num_activities  68  0.190     0.110 -0.451  0.189
f_score_imfts  num_activities  68  0.381     0.321  0.230  0.109
f_score_imfto  num_activities  68  0.212     0.135 -0.398  0.239


--- LOOCV Summary - baseline complexity -controlled ---
                     controls   n  R2_in  adjR2_in     Q2   RMSE
target                                                          
f_score_imtf   num_activities  68  0.462     0.409  0.320  0.131
f_score_imts   num_activities  68  0.308     0.240  0.047  0.129
f_score_imto   num_activities  68  0.380     0.319  0.200  0.191
f_score_imftf  num_activities  68  0.

In [20]:
for t in TRANSLUCENT:
    run_regression(all_df, QUALITY, [*REDUCED_BASELINE, t], OUT_DIR, f"mixed_{t}")
    loocv_summary(all_df, QUALITY, [*REDUCED_BASELINE, t], OUT_DIR, f"mixed_{t}")


--- f_score_imtf ---
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                     1.0318      0.077     13.397      0.000       0.878       1.186
variety                   0.0009      0.002      0.370      0.713      -0.004       0.006
trace_length_avg         -0.0132      0.004     -3.744      0.000      -0.020      -0.006
variant_entropy_norm      0.0183      0.117      0.157      0.876      -0.215       0.252
affinity                 -0.1094      0.086     -1.269      0.209      -0.282       0.063
sequence_entropy_norm    -0.3829      0.167     -2.294      0.025      -0.717      -0.049
complexity_avg_raw       -0.0552      0.015     -3.580      0.001      -0.086      -0.024
n=68  R2=0.555  adj.R2=0.512  F p=0.0000

--- f_score_imts ---
                            coef    std err          t      P>|t|      [0.025      0.975]
---------------

In [22]:
for t in TRANSLUCENT:
    run_regression(all_df, QUALITY, [*REDUCED_BASELINE, t], OUT_DIR, f"mixed_{t}", controls=CONTROL)
    loocv_summary(all_df, QUALITY, [*REDUCED_BASELINE, t], OUT_DIR, f"mixed_{t}", controls=CONTROL)


--- f_score_imtf ---  (controls: num_activities)
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                     1.0129      0.076     13.340      0.000       0.861       1.165
variety                  -0.0078      0.005     -1.538      0.129      -0.018       0.002
trace_length_avg         -0.0125      0.003     -3.621      0.001      -0.019      -0.006
variant_entropy_norm      0.0388      0.115      0.338      0.736      -0.190       0.268
affinity                 -0.1722      0.090     -1.908      0.061      -0.353       0.008
sequence_entropy_norm    -0.3624      0.164     -2.216      0.030      -0.690      -0.035
complexity_avg_raw       -0.0709      0.017     -4.148      0.000      -0.105      -0.037
num_activities            0.0086      0.004      1.950      0.056      -0.000       0.018
n=68  R2=0.582  adj.R2=0.533  F p=0.0000

--- f_sc